# Meta Model

## Problem Definition

**Question.** Given an OOF primary direction, should the strategy act, and at what confidence-derived size?

**Role in the workflow.** Select and tune the secondary act/pass model without changing primary direction.

**Inputs.** Cleaned, weighted events, primary OOF/holdout predictions, primary features, `primary_side`, and `primary_confidence`. `target_return` remains labeling metadata and is not a model input.

**Outputs.** Meta model artifact, candidate/tuning/importance tables, and OOF plus holdout act/probability predictions.

**Why this method.** F1 is primary because the positive class means taking a potentially profitable proposed trade; log loss and precision remain visible.

**Assumptions.** Development meta-labels use only primary OOF predictions; primary direction is immutable; holdout is evaluated once.

**Handoff.** Act/pass and meta probability to `bet_sizing.ipynb`.


## OOF-Only Meta-Label Construction

- For development, `meta_label = 1` only when `primary_side × raw_return > 0`.
- The helper rejects any primary row not explicitly marked `oof`; in-sample primary predictions cannot create meta-labels.
- The meta feature set adds only the OOF primary side and confidence to the same event-start features; raw and target returns remain outcomes, not predictors.
- All four candidate families use random state 42 and five purged folds with a 1% embargo; weighted F1 selects the act/pass family because missed and spurious actions both matter.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, log_loss, precision_score

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.strategy_modeling.cross_validation import PurgedKFold
from src.strategy_modeling.feature_importance import get_orthogonal_features
from src.strategy_modeling.model_workflow import (
    build_candidate_classifiers,
    build_meta_training_frame,
    candidate_parameter_grids,
    generate_oof_predictions,
    get_primary_feature_columns,
    score_binary_predictions,
)

RANDOM_STATE = 42
period = "2025-01-01_2025-12-31"
event_path = PROJECT_ROOT / f"data/research_data/events/aapl_news_modeling_prepared_{period}.parquet"
artifact_dir = PROJECT_ROOT / "data/model_artifact"

events = pd.read_parquet(event_path).set_index("event_start").sort_index()
primary_predictions = pd.read_parquet(artifact_dir / "primary_predictions.parquet").sort_index()
primary_features = get_primary_feature_columns(events.reset_index())

development_primary = primary_predictions[primary_predictions["partition"].eq("development")]
development_events = events.loc[development_primary.index]
primary_oof_contract = development_primary.rename(columns={"primary_side": "prediction", "primary_probability": "probability"})
meta_development = build_meta_training_frame(
    development_events,
    primary_oof_contract[["prediction", "probability", "prediction_source"]],
)

meta_features = [*primary_features, "primary_side", "primary_confidence"]
X_development = meta_development[meta_features]
y_development = meta_development["meta_label"].astype("int8")
w_development = meta_development["sample_weight"].astype(float)
t1_development = meta_development["event_end"]
cv = PurgedKFold(n_splits=5, t1=t1_development, pct_embargo=0.01)

candidates = build_candidate_classifiers(random_state=RANDOM_STATE, n_jobs=1)
comparison_rows = []
for name, estimator in candidates.items():
    predictions = generate_oof_predictions(estimator, X_development, y_development, w_development, cv, positive_label=1)
    scores = score_binary_predictions(
        y_development,
        predictions["prediction"],
        predictions["probability"],
        w_development,
        class_labels=[0, 1],
        positive_label=1,
    )
    comparison_rows.append({
        "candidate": name,
        "f1": scores["f1"],
        "log_loss": scores["log_loss"],
        "precision": scores["precision"],
    })

comparison = pd.DataFrame(comparison_rows).set_index("candidate").sort_values(["f1", "log_loss"], ascending=[False, True])
selected_name = comparison.index[0]
display(pd.Series({"meta_events": len(meta_development), "act_labels": int(y_development.sum()), "pass_labels": int((1 - y_development).sum()), "meta_features": len(meta_features)}, name="value").to_frame())
display(comparison)


/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:919: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:925: RuntimeWarning: invalid value encountered in divide
  oob_decision_function = predictions / predictions.sum(axis=1)[:, np.newaxis]


/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:919: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:925: RuntimeWarning: invalid value encountered in divide
  oob_decision_function = predictions / predictions.sum(axis=1)[:, np.newaxis]
/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:919: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:925: RuntimeWarning: invalid value encountered in divide
  oob_decision_function

,value
meta_events,157
act_labels,88
pass_labels,69
meta_features,55


,f1,log_loss,precision
candidate,,,
random_forest,0.754437,0.599687,0.666531
bagging,0.727696,0.619123,0.680413
gradient_boosting,0.675543,0.801582,0.637824
adaboost,0.657950,0.605403,0.706900


## Purged Tuning and Meta OOF Output

- The selected family is tuned by development F1, with log loss and precision reported.
- Each stored development meta prediction is out-of-fold under the same purging and embargo contract.
- Only the selected meta family is evaluated over its three-value compact grid.
- F1 ranks configurations and log loss breaks equal F1 results, while every saved development action and probability comes from a purged OOF estimator; this keeps tuning tractable without relaxing the provenance requirement.

In [2]:
tuning_rows = []
tuned_oof = {}
for configuration in candidate_parameter_grids()[selected_name]:
    estimator = clone(candidates[selected_name]).set_params(**configuration)
    predictions = generate_oof_predictions(estimator, X_development, y_development, w_development, cv, positive_label=1)
    key = repr(configuration)
    tuned_oof[key] = predictions
    scores = score_binary_predictions(
        y_development,
        predictions["prediction"],
        predictions["probability"],
        w_development,
        class_labels=[0, 1],
        positive_label=1,
    )
    tuning_rows.append({
        "configuration": key,
        **configuration,
        "f1": scores["f1"],
        "log_loss": scores["log_loss"],
        "precision": scores["precision"],
    })

tuning = pd.DataFrame(tuning_rows).sort_values(["f1", "log_loss"], ascending=[False, True], ignore_index=True)
best_configuration = {key: tuning.loc[0, key] for key in candidate_parameter_grids()[selected_name][0]}
tuned_estimator = clone(candidates[selected_name]).set_params(**best_configuration)
meta_oof = tuned_oof[tuning.loc[0, "configuration"]]

meta_oof_output = meta_development[["event_end", "raw_return", "direction_label", "sample_weight", "primary_side", "primary_probability", "primary_confidence", "meta_label"]].copy()
meta_oof_output["partition"] = "development"
meta_oof_output["meta_action"] = meta_oof["prediction"].astype("int8")
meta_oof_output["meta_probability"] = meta_oof["probability"]
meta_oof_output["prediction_source"] = meta_oof["prediction_source"]
meta_oof_output["cv_fold"] = meta_oof["fold"]
display(tuning)
display(meta_oof_output.head())


,configuration,model__max_features,f1,log_loss,precision
0,{'model__max_features': 'sqrt'},sqrt,0.754437,0.599687,0.666531
1,{'model__max_features': 0.5},0.5,0.716140,0.603504,0.670925
2,{'model__max_features': 1.0},1.0,0.698084,0.624611,0.661713


,event_end,raw_return,direction_label,sample_weight,primary_side,primary_probability,primary_confidence,meta_label,partition,meta_action,meta_probability,prediction_source,cv_fold
event_start,,,,,,,,,,,,,
2025-01-13 14:30:01.329809+00:00,2025-01-13 14:34:17.401727+00:00,-0.006778,-1,4.044092,-1,0.450000,0.550000,1,development,1,0.883333,oof,0
2025-01-16 14:30:01.488226+00:00,2025-01-16 14:34:43.996829+00:00,-0.004820,-1,0.484717,1,0.591667,0.591667,0,development,0,0.450000,oof,0
2025-01-17 14:30:01.792073+00:00,2025-01-17 14:39:01.069235+00:00,-0.007032,-1,0.750442,-1,0.275000,0.725000,1,development,1,0.866667,oof,0
2025-01-17 17:20:53.298606+00:00,2025-01-21 14:30:00.854179+00:00,-0.025297,-1,1.049062,-1,0.341667,0.658333,1,development,1,0.783333,oof,0
2025-01-21 14:30:00.854179+00:00,2025-01-21 14:34:53.478136+00:00,-0.007594,-1,1.878389,-1,0.483333,0.516667,1,development,1,0.716667,oof,0


## Development Feature Importance and Error Analysis

- The same MDI, MDA, SFI, and orthogonal views are recomputed for the meta target on development only.
- This distinguishes features useful for direction from features useful for deciding whether to trust that direction.
- As in the primary diagnostic, a separate 120-tree balanced random forest with `sqrt` feature sampling, seed 42, and one worker provides a stable reference.
- Five MDA permutations trade extra computation for less shuffle noise, and the 95% orthogonal threshold reports redundancy without changing the meta feature set.

In [3]:
diagnostic_forest = RandomForestClassifier(
    n_estimators=120,
    class_weight="balanced_subsample",
    max_features="sqrt",
    n_jobs=1,
    random_state=RANDOM_STATE,
).fit(X_development, y_development, sample_weight=w_development)

mdi = pd.Series(diagnostic_forest.feature_importances_, index=meta_features, name="mdi")
mda_result = permutation_importance(
    diagnostic_forest,
    X_development,
    y_development,
    scoring="neg_log_loss",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=1,
    sample_weight=w_development,
)
mda = pd.Series(mda_result.importances_mean, index=meta_features, name="mda")

sfi_scores = {}
for feature in meta_features:
    single_predictions = generate_oof_predictions(candidates["random_forest"], X_development[[feature]], y_development, w_development, cv, positive_label=1)
    scores = score_binary_predictions(
        y_development,
        single_predictions["prediction"],
        single_predictions["probability"],
        w_development,
        class_labels=[0, 1],
        positive_label=1,
    )
    sfi_scores[feature] = -scores["log_loss"]
sfi = pd.Series(sfi_scores, name="sfi_neg_log_loss")

importance = pd.concat([mdi, mda, sfi], axis=1).sort_values("mda", ascending=False)
orthogonal = get_orthogonal_features(X_development, var_thres=0.95)
orthogonal_summary = pd.DataFrame({"component": orthogonal.columns, "label_correlation": [orthogonal[column].corr(y_development) for column in orthogonal.columns]})

importance.to_parquet(artifact_dir / "meta_feature_importance.parquet")
orthogonal_summary.to_parquet(artifact_dir / "meta_orthogonal_features.parquet", index=False)
display(importance.head(10))
display(orthogonal_summary.head())


,mdi,mda,sfi_neg_log_loss
Commodity Channel Index,0.052545,0.053631,-2.600964
TRIX,0.047522,0.045864,-3.511938
MACD Signal Line,0.048437,0.045747,-2.639761
True Range,0.052489,0.043232,-1.015062
Average True Range,0.062661,0.042382,-1.619566
On-Balance Volume,0.031459,0.031510,-1.390737
Force Index,0.046962,0.031338,-2.989552
Ultimate Oscillator,0.032916,0.031043,-3.668562
MACD Line,0.039759,0.030779,-2.547546
Percentage Price Oscillator,0.033961,0.027848,-2.608424


,component,label_correlation
0,PC_1,0.010835
1,PC_2,-0.013383
2,PC_3,-0.175372
3,PC_4,0.032057
4,PC_5,-0.133798


## Final Fit and One-Time Holdout Evaluation

- The final meta estimator fits all OOF-derived development labels.
- Holdout inputs use predictions from the already frozen primary model; holdout outcomes are revealed only to report final metrics.
- The frozen meta estimator fits all development rows whose labels were created from primary OOF predictions.
- Holdout labels are constructed only after fixed actions and probabilities exist, so reporting F1, log loss, and precision cannot feed back into features, thresholds, family choice, or tuning.

In [4]:
final_meta = clone(tuned_estimator).fit(X_development, y_development, sample_weight=w_development.to_numpy())

holdout_primary = primary_predictions[primary_predictions["partition"].eq("holdout")]
holdout_events = events.loc[holdout_primary.index].copy()
holdout_events["primary_side"] = holdout_primary["primary_side"].astype("int8")
holdout_events["primary_probability"] = holdout_primary["primary_probability"]
holdout_events["primary_confidence"] = holdout_primary["primary_confidence"]
holdout_events["meta_label"] = (holdout_events["primary_side"] * holdout_events["raw_return"] > 0).astype("int8")

holdout_probability = final_meta.predict_proba(holdout_events[meta_features])[:, list(final_meta.classes_).index(1)]
holdout_action = final_meta.predict(holdout_events[meta_features]).astype("int8")

meta_holdout_output = holdout_events[["event_end", "raw_return", "direction_label", "sample_weight", "primary_side", "primary_probability", "primary_confidence", "meta_label"]].copy()
meta_holdout_output["partition"] = "holdout"
meta_holdout_output["meta_action"] = holdout_action
meta_holdout_output["meta_probability"] = holdout_probability
meta_holdout_output["prediction_source"] = "holdout"
meta_holdout_output["cv_fold"] = pd.NA

holdout_metrics = pd.Series(
    {
        "f1": f1_score(holdout_events["meta_label"], holdout_action, sample_weight=holdout_events["sample_weight"], zero_division=0),
        "log_loss": log_loss(holdout_events["meta_label"], np.column_stack([1.0 - holdout_probability, holdout_probability]), labels=[0, 1], sample_weight=holdout_events["sample_weight"]),
        "precision": precision_score(holdout_events["meta_label"], holdout_action, sample_weight=holdout_events["sample_weight"], zero_division=0),
    },
    name="holdout",
)

meta_predictions = pd.concat([meta_oof_output, meta_holdout_output]).sort_index()
tuning_artifact = tuning.copy()
for column in tuning_artifact.columns:
    if column.startswith("model__") and tuning_artifact[column].dtype == "object":
        tuning_artifact[column] = tuning_artifact[column].astype(str)
meta_predictions.to_parquet(artifact_dir / "meta_predictions.parquet")
comparison.to_parquet(artifact_dir / "meta_candidate_metrics.parquet")
tuning_artifact.to_parquet(artifact_dir / "meta_tuning_metrics.parquet", index=False)
holdout_metrics.to_frame().to_parquet(artifact_dir / "meta_holdout_metrics.parquet")
joblib.dump(
    {
        "estimator": final_meta,
        "feature_columns": meta_features,
        "selected_candidate": selected_name,
        "best_configuration": best_configuration,
        "random_state": RANDOM_STATE,
    },
    artifact_dir / "meta_model.joblib",
)

display(holdout_metrics.to_frame())
print(artifact_dir / "meta_model.joblib")


,holdout
f1,0.604372
log_loss,0.759585
precision,0.714627


/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/model_artifact/meta_model.joblib


## Results, Limitations, and Handoff

- A meta model may fail to improve primary-only results; that is a valid finding, not a trigger to retune on holdout.
- The action threshold remains the fitted classifier decision and direction always remains `primary_side`.
- The next notebook receives fixed act/pass decisions and meta probabilities.
- No conclusion in this notebook is evidence of live-trading profitability.